In [ ]:
import os
import sys

from corner import corner

sys.path.append('..')

from src.simulator import Model, BurstSimulator
from src.flow_matching.simulator import Model as BatchedModel

from src.c2st import c2st
from src.flow_matching.loader import empty_classifier_from_config, empty_model_from_config, read_config, prob_path_from_config
from src.flow_matching.distributions import UniformPrior, CompositePrior, Posterior, DiscreteUniform
from src.flow_matching.probability_path import GuidedLinearProbabilityPath
from src.flow_matching.integration import EulerODESolver
from src.flow_matching.models import MLPGuidedVectorField, FRBLightCurveCNN, LightCurveThinner, fourier_embedding, LightCurveMLP, UNetEncoder, TransdimensionalModel, EncodedClassifier
from src.flow_matching.transformer import TransformerGuidedField
from src.helpers import record_every, plot_posterior_samples, gen_parameter_labels

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.lines as mlines

import numpy as np
import pandas as pd
import torch
import yaml

from src.flow_matching.plotting import plot_loss, plot_snapshots
from src.flow_matching.helpers import choose_device, build_mlp, find_run_dir

device = choose_device()

# Loading in the FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "15055686" #False
run_dir = None
save_dir = "../checkpoints/"

In [ ]:
# loading the model (via run_id, or path)
run_dir = find_run_dir(job_id, save_dir) if job_id else os.path.join(save_dir, run_dir)

checkpoint_path = os.path.join(run_dir, 'training_checkpoint.pth')
config_path     = os.path.join(run_dir, 'config.yaml')

In [ ]:
# create empty model from config
config = read_config(config_path)
vector_field = empty_model_from_config(config)

In [ ]:
transdimensional = not config['training']['fixed_N'] #False
print(transdimensional)
if transdimensional:
    classifier = empty_classifier_from_config(config)
    vector_field = TransdimensionalModel(classifier, vector_field)

In [ ]:
# load trained model 
print(checkpoint_path)
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Load 'normal' or EMA version 
ema = False
if ema:
    EMA_checkpoint_path = os.path.join(run_dir, "EMA_checkpoint.pth")
    state_dict = torch.load(EMA_checkpoint_path, weights_only=False)
    vector_field.load_state_dict(state_dict) 
else:
    vector_field.load_state_dict(checkpoint["model_state_dict"])

vector_field.eval()
vector_field.to(device)

losses = checkpoint["losses"]

In [ ]:
path = prob_path_from_config(config)
inf_params = config['model']['init_params']['inf_params']
N = path.p_data.model_params['ncomp']
vector_dim = N * len(inf_params)
burstparams = path.p_data.model_params['burstparams']

try:
    mean, std = torch.tensor(config['training']['sample_mean'], device=device), torch.tensor(config['training']['sample_std'], device=device)
except KeyError:
    mean, std = torch.zeros(vector_dim, device=device), torch.ones(vector_dim, device=device)

path.p_data.N_prior.device = device
path.p_data.device = device
path.p_simple.device = device
path.p_data.prior.device = device
path.p_data.prior.set_prior_device()

In [ ]:
# making sure model is valid for magnetar data
if path.p_data.noise != 'poisson':
    raise ValueError('Model not trained with poisson noise.')

# generate FM posterior conditioned on **observed** counts

## Read observational magnetar burst data sample

In [ ]:
filepath = '../' 'observational_data/magnetar_bursts/090122173_+241.347_all_data.dat' # for 2015 comparison
# filepath = '../' 'observational_data/magnetar_bursts/090122173_-000.022_all_data.dat' 
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+139.209_all_data.dat' 
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+037.024_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+147.522_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+271.607_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+290.888_all_data.dat'
# filepath = '../' 'observational_data/magnetar_bursts/090122173_+293.748_all_data.dat'

burst_name = filepath.split('/')[-1]

def read_magnetar_data(filepath):
    df = pd.read_csv(filepath, header=None, delimiter=' ', usecols=[1])
    observed_counts = torch.tensor(df[1], device=device) 
    return observed_counts

observed_counts = read_magnetar_data(filepath)
plt.plot(observed_counts.cpu())
plt.title(f'{burst_name}')
observed_counts.shape

In [ ]:
def downsample(profile, factor):
    """
    Reduce time resolution by merging bins.
    """
    for i in range(factor):
        bins_1 = profile[::2]
        bins_2 = profile[1::2]
        if len(bins_1) != len(bins_2):
            bins_1_new = torch.zeros_like(bins_2)
            bins_1_new = bins_1[:-1]
            bins_1 = bins_1_new
        profile = bins_1 + bins_2
    return profile 

def pad_with_noise(profile, noise_level, noise:str, new_length=1000):
    """Symmetrically pad burst with poisson or gaussian noise.
    """
    assert len(profile) < new_length, 'profile exceeds desired length, padding not possible.'
    assert noise in ['poisson', 'gaussian'], 'noise must be either gaussian or poisson.'

    noise_padding_size = new_length - len(profile)
    padding_left = noise_padding_size // 2 + noise_padding_size % 2
    padding_right = noise_padding_size // 2
    
    if noise == "poisson":
        noise_left = torch.poisson(torch.ones(padding_left, device=device) * noise_level)
        noise_right = torch.poisson(torch.ones(padding_right, device=device) * noise_level)

    elif noise == "gaussian": 
        noise_left = torch.randn(int(padding_left), device=device) * noise_level
        noise_right = torch.randn(int(padding_right), device=device) * noise_level

    prepped_counts = torch.zeros(new_length)
    prepped_counts[:padding_left] = noise_left
    prepped_counts[new_length - padding_right:] = noise_right
    prepped_counts[padding_left:new_length - padding_right] = profile 
    return prepped_counts

In [ ]:
# prep magnetar burst

# reduce time resolution dt 5e-4 -> 1e-3
downsampled = downsample(profile=observed_counts, factor=1)
prepped_counts = pad_with_noise(downsampled, noise_level=3, noise='poisson', new_length=1000)

observed_counts = prepped_counts

plt.plot(prepped_counts.cpu())
plt.title('prepped burst')
plt.show()

In [ ]:
num_samples = 1000  # number of prior samples to transform 
samples_per_batch = 1000
batches = num_samples // samples_per_batch
final_snapshot = torch.zeros((batches * samples_per_batch, vector_dim), device=device)
Ns = torch.zeros((batches * samples_per_batch, 1), device=device)

# use same data point for conditioning all prior samples
condition = torch.tensor(observed_counts / 150, device=device, dtype=torch.float)
simulations = condition.repeat(samples_per_batch, 1)

# initialize ODE solver
solver = EulerODESolver(vector_field)
nts = 200
ts = torch.linspace(0, 1, nts).to(device)

transdimensional=True
if transdimensional:
    # logits = classifier(condition.unsqueeze(0))
    logits = classifier(condition.unsqueeze(0))
    p_N = torch.softmax(logits, dim=1).flatten()
else:
    p_N = torch.zeros(N, device=device)
    p_N[N-1] = 1
    
# integrate in batches
for i in range(batches):
    # simulate ODE starting from x0

    start = i * samples_per_batch
    stop = start + samples_per_batch

    N_samples = torch.multinomial(p_N.repeat(samples_per_batch, 1), num_samples=1) + 1

    x0 = path.p_simple.sample(samples_per_batch, Ns=N_samples).to(device)

    Ns[start:stop, :] = N_samples
    final_snapshot[start:stop, :] = solver.solve(x0, ts.view(1, nts, 1).expand(samples_per_batch, nts, 1), y=simulations, N=N_samples) * std + mean

In [ ]:
final_snapshot

# Classifier p(N)

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

print("Most likely estimate from classifier: ", range(1, len(p_N.cpu().detach().numpy())+1)[torch.where(p_N == torch.max(p_N))[0]])

## posterior samples

In [ ]:
# posterior samples that include all sampled component numbers

# plt.figure(figsize=(15,5))
# plt.subplot(121)
# plt.plot(observed_counts.flatten().cpu(),'k-',alpha=1,  label='noisy flux')

posterior_param_samples = path.p_data.prior.samples_as_dict(final_snapshot)
# Ns_10 = torch.ones_like(Ns) * 2
posterior_curve_samples = BatchedModel(
    device=device, 
    **{
        'time':torch.linspace(0, 1, observed_counts.shape[0]), 
        'burstparams':posterior_param_samples,
        'ybkg':3,
        'ncomp':Ns
        }
        ).get_flux()

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(11, 4))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.5, 1])  # first subplot wider

ax1 = plt.subplot(gs[0])
ax2 = plt.subplot(gs[1])

# plt.tight_layout()
# plt.show() 

ax1.plot(np.linspace(0, 1, observed_counts.shape[0]), observed_counts.flatten().cpu(),'k-', alpha=0.6,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']
# for i in range(1000):
#     random_idx = i#torch.randint(0, num_samples, size=(1,)).item()
#     ax1.plot(
#         np.linspace(0, 1, observed_counts.shape[0]), 
#         posterior_curve_samples[random_idx].cpu(), 
#         alpha=0.07, 
#         color='red',
#         # alpha=1,
#         # color = sample_colors[int(Ns[random_idx].item()) - 1], 
#         label='posterior samples' if i==0 else ''
#         )

# plot mean curve
# mean_profile = torch.median(posterior_curve_samples[0:num_samples], dim=0)[0].cpu()
mean_profile = torch.mean(posterior_curve_samples[0:num_samples], dim=0).cpu()
std_profile  = torch.std(posterior_curve_samples[0:num_samples], dim=0).cpu()

# 95% interval of data
q1 = torch.quantile(posterior_curve_samples, 0.025, dim=0).cpu()
q3 = torch.quantile(posterior_curve_samples, 0.975, dim=0).cpu()
iqr = q3 - q1

# time_ = np.linspace(0, 1, observed_counts.shape[0])
ax1.plot(time_, mean_profile, color='white', label='mean profile', linewidth=1)
# ax1.fill_between(time_, mean_profile - 3*std_profile, mean_profile + 3*std_profile, color='red', alpha=0.7, label='std profile')
ax1.fill_between(time_, q1, q3, color='red', alpha=0.7, label='std profile')

# styling of left plot 
obs_id = filepath.split('/')[-1].split('_')[0]
start_time = filepath.split('/')[-1].split('_')[1]
name_id = obs_id +'_'+start_time
ax1.set_title(name_id, fontsize=16)
# plt.ylim(top=8.5)
# ax1.legend(fontsize=12, loc='upper right')
ax1.grid(linestyle='dotted')
ax1.set_ylabel('counts', fontsize=14)
ax1.set_xlabel('t', fontsize=14)
ax1.set_xlim(0,1)
# ax1.set_ylim(top=11)
ax1.tick_params(axis='both', labelsize=14)
# ax1.set_facecolor('lightgrey')

# plt.subplot(122)

# predicted components probabilities
ax2.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    # color=sample_colors,
    color='lightsteelblue', 
    edgecolor='black',    # color of the bar edge
    linewidth=1,
    label=range(1, 21)
    )
# ax2.legend()
ax2.set_xlabel("$N$", fontsize=14)
ax2.set_ylabel('$p_\phi(N\mid y)$', fontsize=14, labelpad=15)
ax2.tick_params(axis='both', labelsize=14)
ax2.set_xlim(right=20)
# plt.xlim(0, 20)
# plt.grid(axis='y', linestyle='dotted', zorder=-100)

# integer tick labels for component number
import matplotlib.ticker as mticker
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(f'{name_id}.png', dpi=300)
plt.show()


In [ ]:
# similar layout to 2015 paper for comparison
plt.figure(figsize=(6,5.8))
plt.plot(np.linspace(0, 1, observed_counts.shape[0]), observed_counts.flatten().cpu(),'k-', alpha=0.7,  label='observed flux')
sample_colors=['pink', 'yellow', 'orange', 'purple', 'green', 'cyan', 'red', 'grey', 'magenta', 'blue']

# only 10 samples
for i in range(10):
    random_idx = i#torch.randint(0, 10, size=(1,)).item()
    plt.plot(
        np.linspace(0, 1, observed_counts.shape[0]), 
        posterior_curve_samples[random_idx].cpu(), 
        label='posterior samples' if i==0 else ''
        )
# plt.ylim(top=8.5)
# ax1.legend(fontsize=12, loc='upper right')
plt.grid(linestyle='dotted')
plt.ylabel('counts s$^{-1}$', fontsize=14)
plt.xlabel('t', fontsize=14)
plt.xlim(0.1,0.9)
plt.ylim(0)
# ax1.set_ylim(top=11)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.show()

plt.figure(figsize=(6,5.8))
plt.grid(linestyle='dotted', axis='y')

plt.bar(
    range(1, len(p_N.cpu().detach().numpy())+1), 
    height=p_N.cpu().detach().numpy(), 
    # color=sample_colors,
    color='lightsteelblue', 
    edgecolor='black',    # color of the bar edge
    linewidth=1,
    width=0.4,
    zorder=1
    # label=range(1, 21)
    )
# plt.grid(zorder=-1)
plt.ylim(0, 0.205)
plt.xlabel("$N$", fontsize=14)
plt.ylabel('$p_\phi(N\mid y)$', fontsize=14, labelpad=15)
plt.xticks(fontsize=14)
plt.yticks(ticks=[0, 0.04, 0.08, 0.12,0.16,0.2], fontsize=14)
plt.xlim(9, 21)

In [ ]:
if transdimensional:
    MSE_loss = checkpoint['MSE_loss']
    CEL_loss = checkpoint['CEL_loss']
    plt.loglog(MSE_loss, label='MSE')
    plt.loglog(CEL_loss, label='CEL')
    plt.legend()